# Electromagnetic Waves: From Maxwell's Equations to Energy Transport in Matter

This notebook develops the theory of electromagnetic waves in vacuum and in linear media, and
**verifies every algebraic claim symbolically with SymPy**. It is the computational companion to the
essay `maxwell_essay.md`; the code cells are the content of the three verification scripts:

- `maxwell_plane_wave.py` — plane-wave conditions, dispersion relation, polarization, worked example
- `poynting_derivation.py` — Poynting's theorem and plane-wave energy
- `maxwell_matter.py` — Maxwell in matter: damped wave equation, complex dispersion relation, worked examples, energy balance

Conventions: fields are column vectors (`sp.Matrix`); `div`/`curl` are small helpers; the plane-wave
ansatz is $\mathbf{E} = \mathbf{E}_0 e^{i(\mathbf{k}\cdot\mathbf{r}-\omega t)}$ with the physical field its real part.

## 1. Maxwell's equations in vacuum

In vacuum there are no sources ($\rho = 0$, $\mathbf{J} = 0$):

$$\nabla \cdot \mathbf{E} = 0, \qquad \nabla \cdot \mathbf{B} = 0,$$
$$\nabla \times \mathbf{E} = -\frac{\partial \mathbf{B}}{\partial t}, \qquad \nabla \times \mathbf{B} = \mu_0 \varepsilon_0 \frac{\partial \mathbf{E}}{\partial t}.$$

Reading: (i) no electric charge in empty space; (ii) no magnetic monopoles; (iii) Faraday's law — a changing
$\mathbf{B}$ induces a circulating $\mathbf{E}$; (iv) the Ampère–Maxwell law — a changing $\mathbf{E}$
(displacement current) induces a circulating $\mathbf{B}$. The product $\mu_0\varepsilon_0 = 1/c^2$ already hides
the speed of light; the next section shows it is the propagation speed forced by the equations.

## 2. The wave equation

Take the curl of Faraday's law, apply $\nabla\times(\nabla\times\mathbf{E}) = \nabla(\nabla\cdot\mathbf{E}) - \nabla^2\mathbf{E}$,
use $\nabla\cdot\mathbf{E}=0$, and substitute the Ampère–Maxwell law for $\nabla\times\mathbf{B}$:

$$-\nabla^2\mathbf{E} = -\mu_0\varepsilon_0\,\frac{\partial^2\mathbf{E}}{\partial t^2}
\quad\Longrightarrow\quad
\boxed{\;\nabla^2\mathbf{E} = \frac{1}{c^2}\frac{\partial^2\mathbf{E}}{\partial t^2}\;}, \qquad c = \frac{1}{\sqrt{\mu_0\varepsilon_0}}.$$

An identical derivation (curl of Ampère–Maxwell, use $\nabla\cdot\mathbf{B}=0$, substitute Faraday) gives
$\nabla^2\mathbf{B} = \frac{1}{c^2}\partial_t^2 \mathbf{B}$. Both fields propagate through empty space at $c$ —
light is an electromagnetic wave. The next cell verifies the vector identity and the ansatz residual.

In [1]:
import sympy as sp
import cmath

print("SymPy", sp.__version__)

# coordinates / time
x, y, z, t = sp.symbols('x y z t', real=True)

# differential operators on column-vector fields
def div(F):
    return sp.diff(F[0], x) + sp.diff(F[1], y) + sp.diff(F[2], z)

def curl(F):
    return sp.Matrix([sp.diff(F[2], y) - sp.diff(F[1], z),
                      sp.diff(F[0], z) - sp.diff(F[2], x),
                      sp.diff(F[1], x) - sp.diff(F[0], y)])
print("helpers div/curl defined")

SymPy 1.14.0
helpers div/curl defined


In [2]:
# --- verify the wave-equation machinery ---
# (i) vector identity  curl(curl E) = grad(div E) - laplacian(E)  for a generic field
Ex = sp.Function('Ex')(x, y, z, t); Ey = sp.Function('Ey')(x, y, z, t); Ez = sp.Function('Ez')(x, y, z, t)
Eg = sp.Matrix([Ex, Ey, Ez])
divE = div(Eg)
grad_div = sp.Matrix([sp.diff(divE, v) for v in (x, y, z)])
lapEg  = sp.Matrix([sum(sp.diff(Eg[i], v, 2) for v in (x, y, z)) for i in range(3)])
print("curl(curl E) - (grad div E - lap E) =", sp.simplify(curl(curl(Eg)) - (grad_div - lapEg)).T)

# (ii) plane-wave ansatz satisfies  lap E = (1/c^2) d2E/dt2  iff  k^2 = w^2/c^2
eps0, mu0 = sp.symbols('epsilon_0 mu_0', positive=True)
c  = 1/sp.sqrt(mu0*eps0)
kx, ky, kz = sp.symbols('k_x k_y k_z', real=True)
w  = sp.symbols('omega', positive=True)
E0 = sp.Matrix(sp.symbols('E0x E0y E0z', real=True))
f  = sp.exp(sp.I*(kx*x + ky*y + kz*z - w*t))
E  = E0*f
lapE = sp.Matrix([sum(sp.diff(E[i], v, 2) for v in (x, y, z)) for i in range(3)])
res = sp.simplify((lapE - sp.diff(E, t, 2)/c**2)/f)
print("wave-equation residual /f =", res.T)
print("=> vanishes iff  k^2 =", sp.simplify(w**2/c**2))

curl(curl E) - (grad div E - lap E) = Matrix([[0, 0, 0]])


wave-equation residual /f = Matrix([[E0x*(epsilon_0*mu_0*omega**2 - k_x**2 - k_y**2 - k_z**2), E0y*(epsilon_0*mu_0*omega**2 - k_x**2 - k_y**2 - k_z**2), E0z*(epsilon_0*mu_0*omega**2 - k_x**2 - k_y**2 - k_z**2)]])
=> vanishes iff  k^2 = epsilon_0*mu_0*omega**2


## 3. The plane-wave solution

Ansatz $\mathbf{E} = \mathbf{E}_0 e^{i(\mathbf{k}\cdot\mathbf{r}-\omega t)}$, $\mathbf{B} = \mathbf{B}_0 e^{i(\mathbf{k}\cdot\mathbf{r}-\omega t)}$.
Stripping the common factor, each operator becomes a multiplication:

| Equation | Condition |
|---|---|
| $\nabla\cdot\mathbf{E}=0$ | $\mathbf{k}\cdot\mathbf{E}_0 = 0$ |
| $\nabla\cdot\mathbf{B}=0$ | $\mathbf{k}\cdot\mathbf{B}_0 = 0$ |
| Faraday | $\mathbf{B}_0 = \mathbf{k}\times\mathbf{E}_0/\omega$ |
| Ampère–Maxwell | $\mathbf{k}\times\mathbf{B}_0 = -\mu_0\varepsilon_0\omega\,\mathbf{E}_0$ |

Substituting $\mathbf{B}_0$ into the last row and using $\mathbf{k}\times(\mathbf{k}\times\mathbf{E}_0) = \mathbf{k}(\mathbf{k}\cdot\mathbf{E}_0) - k^2\mathbf{E}_0$
leaves $\mathbf{k}(\mathbf{k}\cdot\mathbf{E}_0) + (\mu_0\varepsilon_0\omega^2 - k^2)\mathbf{E}_0 = 0$, i.e. the
**dispersion relation** $\omega = c|\mathbf{k}|$ plus **transversality** $\mathbf{k}\cdot\mathbf{E}_0 = 0$.
From $\mathbf{B}_0 = \mathbf{k}\times\mathbf{E}_0/\omega$ it follows that $\mathbf{E}_0 \perp \mathbf{B}_0$,
$|\mathbf{B}_0| = |\mathbf{E}_0|/c$, and $\{\mathbf{E}_0, \mathbf{B}_0, \hat{\mathbf{k}}\}$ is a right-handed triad.

In [3]:
# --- plane wave in vacuum: strip the common factor exp(i(k.r - wt)) ---
kx, ky, kz = sp.symbols('k_x k_y k_z', real=True)
w = sp.symbols('omega', positive=True)
Ex0, Ey0, Ez0 = sp.symbols('E0x E0y E0z', real=True)
Bx0, By0, Bz0 = sp.symbols('B0x B0y B0z', real=True)

theta = kx*x + ky*y + kz*z - w*t
f = sp.exp(sp.I*theta)
E = sp.Matrix([Ex0, Ey0, Ez0])*f
B = sp.Matrix([Bx0, By0, Bz0])*f
k  = sp.Matrix([kx, ky, kz])
E0 = sp.Matrix([Ex0, Ey0, Ez0])
B0 = sp.Matrix([Bx0, By0, Bz0])

print("div E /f        =", sp.simplify(div(E)/f))
print("div B /f        =", sp.simplify(div(B)/f))
print("curl E /f       =", sp.simplify(curl(E)/f).T)
print("curl B /f       =", sp.simplify(curl(B)/f).T)
print("dE/dt /f        =", sp.simplify(sp.diff(E, t)/f).T)
print("dB/dt /f        =", sp.simplify(sp.diff(B, t)/f).T)

c1 = k.dot(E0)      # k . E0
c2 = k.dot(B0)      # k . B0
far = k.cross(E0) - w*B0           # Faraday  -> k x E0 = w B0
amp = k.cross(B0) + mu0*eps0*w*E0  # Ampere   -> k x B0 = -mu0 eps0 w E0
print("\nFaraday residual  k x E0 - w B0         =", far.T)
print("Ampere  residual  k x B0 + mu0 eps0 w E0 =", amp.T)

div E /f        = I*(E0x*k_x + E0y*k_y + E0z*k_z)
div B /f        = I*(B0x*k_x + B0y*k_y + B0z*k_z)
curl E /f       = Matrix([[I*(-E0y*k_z + E0z*k_y), I*(E0x*k_z - E0z*k_x), I*(-E0x*k_y + E0y*k_x)]])


curl B /f       = Matrix([[I*(-B0y*k_z + B0z*k_y), I*(B0x*k_z - B0z*k_x), I*(-B0x*k_y + B0y*k_x)]])
dE/dt /f        = Matrix([[-I*E0x*omega, -I*E0y*omega, -I*E0z*omega]])
dB/dt /f        = Matrix([[-I*B0x*omega, -I*B0y*omega, -I*B0z*omega]])

Faraday residual  k x E0 - w B0         = Matrix([[-B0x*omega - E0y*k_z + E0z*k_y, -B0y*omega + E0x*k_z - E0z*k_x, -B0z*omega - E0x*k_y + E0y*k_x]])
Ampere  residual  k x B0 + mu0 eps0 w E0 = Matrix([[-B0y*k_z + B0z*k_y + E0x*epsilon_0*mu_0*omega, B0x*k_z - B0z*k_x + E0y*epsilon_0*mu_0*omega, -B0x*k_y + B0y*k_x + E0z*epsilon_0*mu_0*omega]])


In [4]:
# --- substitute B0 = (k x E0)/w into the Ampere residual: two conditions remain ---
res = sp.expand(amp.subs(B0, k.cross(E0)/w))
print("Ampere residual with B0 = (k x E0)/w:", res.T)
k2 = kx**2 + ky**2 + kz**2
factored = sp.expand((c1/w)*k + (mu0*eps0*w - k2/w)*E0)
print("identity check (diff of the two forms):", sp.simplify(res - factored).T)
print("\n=> two conditions remain:")
print("   (1) k.E0 = 0")
print("   (2) mu0*eps0*w - k^2/w = 0   =>   k^2 =", sp.simplify(mu0*eps0*w**2))

# --- polarization consequences (all follow from B0 = k x E0 / w) ---
B0sol = k.cross(E0)/w
print("\nE0 . B0  =", sp.simplify(E0.dot(B0sol)), "  (triple product, two equal vectors)")
print("with k.E0=0: |k x E0|^2 = k^2|E0|^2 - (k.E0)^2 =", sp.simplify(k2*(E0.dot(E0)) - c1**2))

Ampere residual with B0 = (k x E0)/w: Matrix([[-B0y*k_z + B0z*k_y + E0x*epsilon_0*mu_0*omega, B0x*k_z - B0z*k_x + E0y*epsilon_0*mu_0*omega, -B0x*k_y + B0y*k_x + E0z*epsilon_0*mu_0*omega]])
identity check (diff of the two forms): Matrix([[(E0x*k_y**2 + E0x*k_z**2 - E0y*k_x*k_y - E0z*k_x*k_z + omega*(-B0y*k_z + B0z*k_y))/omega, (-E0x*k_x*k_y + E0y*k_x**2 + E0y*k_z**2 - E0z*k_y*k_z + omega*(B0x*k_z - B0z*k_x))/omega, (-E0x*k_x*k_z - E0y*k_y*k_z + E0z*k_x**2 + E0z*k_y**2 + omega*(-B0x*k_y + B0y*k_x))/omega]])

=> two conditions remain:
   (1) k.E0 = 0
   (2) mu0*eps0*w - k^2/w = 0   =>   k^2 = epsilon_0*mu_0*omega**2

E0 . B0  = 0   (triple product, two equal vectors)


with k.E0=0: |k x E0|^2 = k^2|E0|^2 - (k.E0)^2 =

 (E0x**2 + E0y**2 + E0z**2)*(k_x**2 + k_y**2 + k_z**2) - (E0x*k_x + E0y*k_y + E0z*k_z)**2


In [5]:
# --- concrete example: k = k zhat, E0 = Ex xhat, w = c k ---
cval = 1/sp.sqrt(mu0*eps0)
kv   = sp.symbols('k', positive=True)
Exv  = sp.symbols('Ex', positive=True)
E0v  = sp.Matrix([Exv, 0, 0])
B0v  = sp.simplify((sp.Matrix([0, 0, kv]).cross(E0v))/(cval*kv))
print("Example: k=(0,0,k), E0=(Ex,0,0), w=c k  =>  B0 =", B0v.T)
sub = {kx:0, ky:0, kz:kv, w:cval*kv, Ex0:E0v[0], Ey0:0, Ez0:0,
       Bx0:B0v[0], By0:B0v[1], Bz0:B0v[2]}
print("\nVerify all four equations (each should be 0):")
print("  div E /f                  =", sp.simplify(div(E).subs(sub)/f))
print("  div B /f                  =", sp.simplify(div(B).subs(sub)/f))
print("  (curl E + dB/dt)/f        =", sp.simplify((curl(E) + sp.diff(B, t)).subs(sub)/f).T)
print("  (curl B - mu0 eps0 dE/dt) =", sp.simplify((curl(B) - mu0*eps0*sp.diff(E, t)).subs(sub)/f).T)
print("\nE x B /f (Poynting direction) =", sp.simplify(E.cross(B).subs(sub)/f).T)

Example: k=(0,0,k), E0=(Ex,0,0), w=c k  =>  B0 = Matrix([[0, Ex*sqrt(epsilon_0)*sqrt(mu_0), 0]])

Verify all four equations (each should be 0):
  div E /f                  = 0
  div B /f                  = 0
  (curl E + dB/dt)/f        = Matrix([[0, 0, 0]])
  (curl B - mu0 eps0 dE/dt) = Matrix([[0, 0, 0]])

E x B /f (Poynting direction) = Matrix([[0, 0, Ex**2*sqrt(epsilon_0)*sqrt(mu_0)*exp(I*(-k_x*x - k_y*y - k_z*z + omega*t + 2*k*(sqrt(epsilon_0)*sqrt(mu_0)*z - t)/(sqrt(epsilon_0)*sqrt(mu_0))))]])


## 4. Energy density and the Poynting vector

**Poynting's theorem.** Writing $\mathbf{J}$ from the Ampère–Maxwell law, the power per unit volume delivered to the
charges is $\mathbf{J}\cdot\mathbf{E}$. Using the identity $(\nabla\times\mathbf{B})\cdot\mathbf{E} = \nabla\cdot(\mathbf{E}\times\mathbf{B}) + (\nabla\times\mathbf{E})\cdot\mathbf{B}$
and Faraday's law, together with $\mathbf{E}\cdot\partial_t\mathbf{E} = \tfrac12\partial_t E^2$ (and its magnetic analogue), one obtains the local conservation law

$$\boxed{\;\frac{\partial u}{\partial t} + \nabla\cdot\mathbf{S} = -\mathbf{J}\cdot\mathbf{E}, \qquad
u = \frac{\varepsilon_0}{2}|\mathbf{E}|^2 + \frac{1}{2\mu_0}|\mathbf{B}|^2, \qquad
\mathbf{S} = \frac{1}{\mu_0}\mathbf{E}\times\mathbf{B}\;}$$

— field energy changes by the flux $\mathbf{S}$ through the boundary plus the work $\mathbf{J}\cdot\mathbf{E}$ done on the charges.
In vacuum, $\partial_t u + \nabla\cdot\mathbf{S} = 0$.

**Plane wave.** For $\mathbf{E} = E_0\cos\varphi\,\hat{x}$, $\mathbf{B} = \frac{E_0}{c}\cos\varphi\,\hat{y}$: the electric and
magnetic contributions to $u$ are *equal point by point*, $u = \varepsilon_0 E_0^2 \cos^2\varphi$, $\mathbf{S} = cu\,\hat{z}$,
and the cycle averages give the **intensity** $I = \langle S\rangle = \dfrac{E_0^2}{2\mu_0 c} = \dfrac{c\varepsilon_0 E_0^2}{2}$.

In [6]:
# --- Poynting's theorem in vacuum, for GENERIC fields ---
# J from Ampere-Maxwell:  J = (1/mu0)(curl B - mu0 eps0 dE/dt);  J.E
Ex = sp.Function('Ex')(x, y, z, t); Ey = sp.Function('Ey')(x, y, z, t); Ez = sp.Function('Ez')(x, y, z, t)
Bx = sp.Function('Bx')(x, y, z, t); By = sp.Function('By')(x, y, z, t); Bz = sp.Function('Bz')(x, y, z, t)
E = sp.Matrix([Ex, Ey, Ez]); B = sp.Matrix([Bx, By, Bz])
dE = sp.diff(E, t); dB = sp.diff(B, t)

JdotE = sp.expand((curl(B) - mu0*eps0*dE).dot(E)/mu0)

# energy density and the Poynting combination
u = eps0*E.dot(E)/2 + B.dot(B)/(2*mu0)
comb = sp.expand(JdotE + sp.diff(u, t) + div(E.cross(B))/mu0)
faraday_factor = sp.expand((curl(E) + dB).dot(B)/mu0)
print("J.E + du/dt + div(E x B)/mu0         =", comb)
print("Faraday factor (curl E + dB/dt).B/mu0 =", faraday_factor)
print("combination == Faraday factor ?", sp.simplify(comb - faraday_factor) == 0)
print("=> under Faraday (curl E = -dB/dt):  J.E + du/dt + div(E x B)/mu0 = 0")
print("   i.e.  du/dt + div(S) = -J.E,   S = E x B / mu0   (Poynting theorem)")
print("   vacuum (J=0):                 du/dt + div(S) = 0")

J.E + du/dt + div(E x B)/mu0         =

 Bx(x, y, z, t)*Derivative(Bx(x, y, z, t), t)/mu_0 - Bx(x, y, z, t)*Derivative(Ey(x, y, z, t), z)/mu_0 + Bx(x, y, z, t)*Derivative(Ez(x, y, z, t), y)/mu_0 + By(x, y, z, t)*Derivative(By(x, y, z, t), t)/mu_0 + By(x, y, z, t)*Derivative(Ex(x, y, z, t), z)/mu_0 - By(x, y, z, t)*Derivative(Ez(x, y, z, t), x)/mu_0 + Bz(x, y, z, t)*Derivative(Bz(x, y, z, t), t)/mu_0 - Bz(x, y, z, t)*Derivative(Ex(x, y, z, t), y)/mu_0 + Bz(x, y, z, t)*Derivative(Ey(x, y, z, t), x)/mu_0
Faraday factor (curl E + dB/dt).B/mu0 = Bx(x, y, z, t)*Derivative(Bx(x, y, z, t), t)/mu_0 - Bx(x, y, z, t)*Derivative(Ey(x, y, z, t), z)/mu_0 + Bx(x, y, z, t)*Derivative(Ez(x, y, z, t), y)/mu_0 + By(x, y, z, t)*Derivative(By(x, y, z, t), t)/mu_0 + By(x, y, z, t)*Derivative(Ex(x, y, z, t), z)/mu_0 - By(x, y, z, t)*Derivative(Ez(x, y, z, t), x)/mu_0 + Bz(x, y, z, t)*Derivative(Bz(x, y, z, t), t)/mu_0 - Bz(x, y, z, t)*Derivative(Ex(x, y, z, t), y)/mu_0 + Bz(x, y, z, t)*Derivative(Ey(x, y, z, t), x)/mu_0
combination == Faraday fact

In [7]:
# --- plane wave: E = E0 cos(kz - wt) xhat, B = (E0/c) cos(kz - wt) yhat ---
c = 1/sp.sqrt(mu0*eps0)
k, w, E0 = sp.symbols('k omega E0', positive=True)
w = c*k                      # dispersion relation
phi = k*z - w*t
Ev = sp.Matrix([E0*sp.cos(phi), 0, 0])
Bv = sp.Matrix([0, (E0/c)*sp.cos(phi), 0])
Sv = Ev.cross(Bv)/mu0

uE = eps0*Ev.dot(Ev)/2
uB = Bv.dot(Bv)/(2*mu0)
uv = uE + uB

print("uE (electric)   =", sp.simplify(uE))
print("uB (magnetic)   =", sp.simplify(uB))
print("uE == uB        ?", sp.simplify(uE - uB) == 0)
print("u = uE + uB     =", sp.simplify(uv))
print("S               =", sp.simplify(Sv.T))
print("du/dt + div S   =", sp.simplify(sp.diff(uv, t) + sp.diff(Sv[0], x) + sp.diff(Sv[1], y) + sp.diff(Sv[2], z)))
print("S - c*u*zhat    =", sp.simplify((Sv - c*uv*sp.Matrix([0, 0, 1])).T))

# cycle averages over one period T = 2 pi / omega
T = 2*sp.pi/w
u_avg = sp.integrate(uv, (t, 0, T))/T
S_avg = sp.integrate(Sv[2], (t, 0, T))/T
print("<u> over period    =", sp.simplify(u_avg))
print("<S_z> over period  =", sp.simplify(S_avg))
print("<S> == c <u> zhat ?", sp.simplify(S_avg - c*u_avg) == 0)
print("I = <S>            =", sp.simplify(S_avg), "  (i.e. E0^2/(2 mu0 c) = c eps0 E0^2/2)")

uE (electric)   = E0**2*epsilon_0*cos(k*(z - t/(sqrt(epsilon_0)*sqrt(mu_0))))**2/2
uB (magnetic)   = E0**2*epsilon_0*cos(k*(z - t/(sqrt(epsilon_0)*sqrt(mu_0))))**2/2
uE == uB        ? True


u = uE + uB     = E0**2*epsilon_0*cos(k*(z - t/(sqrt(epsilon_0)*sqrt(mu_0))))**2
S               = Matrix([[0, 0, E0**2*sqrt(epsilon_0)*cos(k*(z - t/(sqrt(epsilon_0)*sqrt(mu_0))))**2/sqrt(mu_0)]])
du/dt + div S   = 0
S - c*u*zhat    = Matrix([[0, 0, 0]])


<u> over period    = E0**2*epsilon_0/2
<S_z> over period  = E0**2*sqrt(epsilon_0)/(2*sqrt(mu_0))
<S> == c <u> zhat ? True
I = <S>            = E0**2*sqrt(epsilon_0)/(2*sqrt(mu_0))   (i.e. E0^2/(2 mu0 c) = c eps0 E0^2/2)


## 5. Maxwell's equations in matter

Macroscopic fields $\mathbf{D},\mathbf{H}$ with free sources; linear, isotropic, homogeneous Ohmic medium:

$$\mathbf{D} = \varepsilon\,\mathbf{E}, \qquad \mathbf{B} = \mu\,\mathbf{H}, \qquad \mathbf{J}_f = \sigma\,\mathbf{E}.$$

**Poynting's theorem in matter** (same derivation, with $\mathbf{H},\mathbf{D}$ in place of $\mathbf{B}/\mu_0,\ \varepsilon_0\mathbf{E}$):

$$\frac{\partial u}{\partial t} + \nabla\cdot\mathbf{S} = -\mathbf{J}_f\cdot\mathbf{E}, \qquad
u = \tfrac12\left(\mathbf{E}\cdot\mathbf{D} + \mathbf{B}\cdot\mathbf{H}\right), \qquad \mathbf{S} = \mathbf{E}\times\mathbf{H},$$

with Ohmic loss $\mathbf{J}_f\cdot\mathbf{E} = \sigma|\mathbf{E}|^2 \ge 0$.

**Damped wave equation** (curl Faraday, substitute Ampère with the constitutive relations, $\nabla\cdot\mathbf{E}=0$):

$$\nabla^2\mathbf{E} = \mu\varepsilon\,\frac{\partial^2\mathbf{E}}{\partial t^2} + \mu\sigma\,\frac{\partial\mathbf{E}}{\partial t}.$$

**Complex dispersion relation** for the plane-wave ansatz:

$$\mathbf{k}^2 = \mu\varepsilon\omega^2 + i\,\mu\sigma\omega, \qquad \kappa = \beta + i\alpha:\quad
\beta^2 - \alpha^2 = \mu\varepsilon\omega^2,\ \qquad 2\alpha\beta = \mu\sigma\omega.$$

$\beta$ sets the phase velocity $v_p = \omega/\beta$; $\alpha$ is the attenuation (skin depth $\delta = 1/\alpha$);
the intrinsic impedance $\eta = \omega\mu/\kappa = \sqrt{\mu/(\varepsilon - i\sigma/\omega)}$ is complex when $\sigma \neq 0$.
**Good-conductor limit** ($\sigma \gg \omega\varepsilon$): $\delta = \sqrt{2/(\mu\sigma\omega)}$, $\eta \approx \sqrt{\omega\mu/\sigma}\,e^{-i\pi/4}$.

In [8]:
# --- matter: linear isotropic homogeneous Ohmic medium ---
#     D = eps E,  B = mu H,  J_f = sigma E ;  source-free (rho_f = 0)
x, y, z, t = sp.symbols('x y z t', real=True)
eps, mu, sig = sp.symbols('epsilon mu sigma', positive=True)
eps0, mu0 = sp.symbols('epsilon_0 mu_0', positive=True)
E0x, E0y, E0z = sp.symbols('E0x E0y E0z', real=True)
kx, ky, kz, w = sp.symbols('k_x k_y k_z omega', real=True)

# ---------- Part 0: Poynting theorem in matter ----------
Ex = sp.Function('Ex')(x, y, z, t); Ey = sp.Function('Ey')(x, y, z, t); Ez = sp.Function('Ez')(x, y, z, t)
Hx = sp.Function('Hx')(x, y, z, t); Hy = sp.Function('Hy')(x, y, z, t); Hz = sp.Function('Hz')(x, y, z, t)
Ev = sp.Matrix([Ex, Ey, Ez]); Hv = sp.Matrix([Hx, Hy, Hz])
Dv = eps*Ev; Bv = mu*Hv
Jf = curl(Hv) - sp.diff(Dv, t)              # J_f from Ampere
u  = sp.Rational(1, 2)*(Ev.dot(Dv) + Bv.dot(Hv))
S  = Ev.cross(Hv)
faraday = (curl(Ev) + sp.diff(Bv, t)).dot(Hv)
comb = sp.expand(sp.diff(u, t) + div(S) + Jf.dot(Ev) - faraday)
print("du/dt + div(E x H) + Jf.E - (curl E + dB/dt).H  =", sp.simplify(comb))
print("=> on solutions:  du/dt + div S = -Jf.E,   u = 1/2(E.D + B.H),   S = E x H")
print("   ohmic loss:  Jf.E = sigma |E|^2 >= 0  (Joule heating)")

du/dt + div(E x H) + Jf.E - (curl E + dB/dt).H  =

 0
=> on solutions:  du/dt + div S = -Jf.E,   u = 1/2(E.D + B.H),   S = E x H
   ohmic loss:  Jf.E = sigma |E|^2 >= 0  (Joule heating)


In [9]:
# ---------- Part 1: plane wave, algebraic conditions (lossy) ----------
f = sp.exp(sp.I*(kx*x + ky*y + kz*z - w*t))
E0 = sp.Matrix([E0x, E0y, E0z]); k = sp.Matrix([kx, ky, kz])
E = E0*f
B0 = k.cross(E0)/w                        # from Faraday: k x E0 = w B0
B = B0*f
H = B/mu; D = eps*E
k2 = kx**2 + ky**2 + kz**2

print("div D /f  =", sp.simplify(div(D)/f))
print("div B /f  =", sp.simplify(div(B)/f))
print("Faraday residual (B0 = k x E0/w) /f =", sp.simplify((curl(E) + sp.diff(B, t))/f).T)
amp_res = sp.simplify((curl(H) - sp.diff(D, t) - sig*E)/f)
print("Ampere residual (with B0 = k x E0/w) /f =", amp_res.T)
cand = (sp.I*(k.dot(E0))/(mu*w))*k + (sp.I*(mu*eps*w**2 - k2)/(mu*w) - sig)*E0
print("identity check (residual - candidate) =", sp.simplify(amp_res - cand).T)
print("=> conditions:  (1) k.E0 = 0   (2) k^2 = mu*eps*w^2 + i*mu*sigma*w")

div D /f  = I*epsilon*(E0x*k_x + E0y*k_y + E0z*k_z)
div B /f  = 0


Faraday residual (B0 = k x E0/w) /f = Matrix([[0, 0, 0]])


Ampere residual (with B0 = k x E0/w) /f = Matrix([[(-E0x*mu*omega*(-I*epsilon*omega + sigma) - I*k_y*(E0x*k_y - E0y*k_x) - I*k_z*(E0x*k_z - E0z*k_x))/(mu*omega), (E0y*mu*omega*(I*epsilon*omega - sigma) + I*k_x*(E0x*k_y - E0y*k_x) - I*k_z*(E0y*k_z - E0z*k_y))/(mu*omega), (E0z*mu*omega*(I*epsilon*omega - sigma) + I*k_x*(E0x*k_z - E0z*k_x) + I*k_y*(E0y*k_z - E0z*k_y))/(mu*omega)]])
identity check (residual - candidate) = Matrix([[0, 0, 0]])
=> conditions:  (1) k.E0 = 0   (2) k^2 = mu*eps*w^2 + i*mu*sigma*w


In [10]:
# ---------- Part 2: damped wave equation, ansatz ----------
lapE = sp.Matrix([sum(sp.diff(E[i], v, 2) for v in (x, y, z)) for i in range(3)])
res2 = sp.simplify((lapE - eps*mu*sp.diff(E, t, 2) - mu*sig*sp.diff(E, t))/f)
print("damped wave equation  lap E = mu eps d2E/dt2 + mu sigma dE/dt")
print("ansatz residual /f  =", sp.simplify(res2).T)
print("lossless (sigma=0)  =", sp.simplify(res2.subs(sig, 0)).T)

damped wave equation  lap E = mu eps d2E/dt2 + mu sigma dE/dt
ansatz residual /f  = Matrix([[E0x*(epsilon*mu*omega**2 - k_x**2 - k_y**2 - k_z**2 + I*mu*omega*sigma), E0y*(epsilon*mu*omega**2 - k_x**2 - k_y**2 - k_z**2 + I*mu*omega*sigma), E0z*(epsilon*mu*omega**2 - k_x**2 - k_y**2 - k_z**2 + I*mu*omega*sigma)]])
lossless (sigma=0)  = Matrix([[E0x*(epsilon*mu*omega**2 - k_x**2 - k_y**2 - k_z**2), E0y*(epsilon*mu*omega**2 - k_x**2 - k_y**2 - k_z**2), E0z*(epsilon*mu*omega**2 - k_x**2 - k_y**2 - k_z**2)]])


In [11]:
# ---------- Part 3a: lossless plane wave, n = 2 glass (physical real fields) ----------
c = 1/sp.sqrt(mu0*eps0)
eps_g = 4*eps0; mu_g = mu0
v = 1/sp.sqrt(eps_g*mu_g)
kv, Exv = sp.symbols('k E0x', positive=True)
wv = v*kv
phi = kv*z - wv*t
Eg = sp.Matrix([Exv*sp.cos(phi), 0, 0])
Bg = sp.Matrix([0, (Exv/v)*sp.cos(phi), 0])      # B0 = E0/v
Hg = Bg/mu_g
Dg = eps_g*Eg
print("Part 3a: lossless, eps=4 eps0, mu=mu0  (n = 2, v = c/2)")
print("v =", sp.simplify(v), "   B0 = E0/v =", sp.simplify(Exv/v), "   eta = sqrt(mu/eps) =", sp.simplify(sp.sqrt(mu_g/eps_g)))
# verify all four on the real fields
print("verify (all should be 0):")
print("  div D               =", sp.simplify(div(Dg)))
print("  div B               =", sp.simplify(div(Bg)))
print("  (curl E + dB/dt)    =", sp.simplify((curl(Eg) + sp.diff(Bg, t))).T)
print("  (curl H - dD/dt)    =", sp.simplify((curl(Hg) - sp.diff(Dg, t))).T)
ug = sp.Rational(1, 2)*(Eg.dot(Dg) + Bg.dot(Hg))
Sg = Eg.cross(Hg)
T = 2*sp.pi/wv
print("u(t)          =", sp.simplify(ug))
print("S(t)          =", sp.simplify(Sg.T))
uav = sp.integrate(ug, (t, 0, T))/T
Sav = sp.integrate(Sg[2], (t, 0, T))/T
print("<u>           =", sp.simplify(uav))
print("<S_z>         =", sp.simplify(Sav))
print("<S> == v <u> zhat ?", sp.simplify(Sav - v*uav) == 0)
print("<S> == E0^2/(2 eta) ?", sp.simplify(Sav - Exv**2/(2*sp.sqrt(mu_g/eps_g))) == 0)
print("<S> == (1/2) eps v E0^2 ?", sp.simplify(Sav - sp.Rational(1,2)*eps_g*v*Exv**2) == 0)

Part 3a: lossless, eps=4 eps0, mu=mu0  (n = 2, v = c/2)
v = 1/(2*sqrt(epsilon_0)*sqrt(mu_0))    B0 = E0/v = 2*E0x*sqrt(epsilon_0)*sqrt(mu_0)    eta = sqrt(mu/eps) = sqrt(mu_0)/(2*sqrt(epsilon_0))
verify (all should be 0):
  div D               = 0
  div B               = 0
  (curl E + dB/dt)    = Matrix([[0, 0, 0]])
  (curl H - dD/dt)    = Matrix([[0, 0, 0]])


u(t)          = 4*E0x**2*epsilon_0*cos(k*(z - t/(2*sqrt(epsilon_0)*sqrt(mu_0))))**2
S(t)          = Matrix([[0, 0, 2*E0x**2*sqrt(epsilon_0)*cos(k*(z - t/(2*sqrt(epsilon_0)*sqrt(mu_0))))**2/sqrt(mu_0)]])


<u>           = 2*E0x**2*epsilon_0
<S_z>         = E0x**2*sqrt(epsilon_0)/sqrt(mu_0)
<S> == v <u> zhat ? True
<S> == E0^2/(2 eta) ? True


<S> == (1/2) eps v E0^2 ? True


In [12]:
# ---------- Part 3b: lossy, copper 60 Hz (numeric) ----------
import cmath
mu0n = 4e-7*cmath.pi; eps0n = 8.8541878128e-12
sigc = 5.8e7; wn = 2*cmath.pi*60.0
k2n = eps0n*mu0n*wn**2 + 1j*mu0n*sigc*wn
kn = cmath.sqrt(k2n)                       # branch: Re>0, Im>0
beta, alpha = kn.real, kn.imag
print("Part 3b: copper, sigma=5.8e7 S/m, mu=mu0, eps=eps0, f=60 Hz")
print("k^2 = mu eps w^2 + i mu sigma w   (complex)")
print("beta = %.4g   alpha = %.4g   (1/m)" % (beta, alpha))
print("skin depth delta = 1/alpha = %.4g m" % (1/alpha))
print("good-conductor approx: beta=alpha=sqrt(mu sigma w/2) = %.4g,  delta = sqrt(2/(mu sigma w)) = %.4g m"
      % (cmath.sqrt(mu0n*sigc*wn/2).real, cmath.sqrt(2/(mu0n*sigc*wn)).real))
print("check (beta+i alpha)^2 == k^2 :", abs((beta+1j*alpha)**2 - k2n) < 1e-8*abs(k2n))
print("check 2 alpha beta == mu sigma w :", abs(2*alpha*beta - mu0n*sigc*wn) < 1e-12*mu0n*sigc*wn)
etac = wn*mu0n/(beta+1j*alpha)
etac_gc = cmath.sqrt(wn*mu0n/sigc)*cmath.exp(-1j*cmath.pi/4)
print("eta_c = w mu/k = %.4g * exp(i %.4g pi)  |eta_c| = %.4g ohm" % (abs(etac), cmath.phase(etac)/cmath.pi, abs(etac)))
print("good-conductor eta_c = sqrt(w mu/sigma) e^{-i pi/4} = %.4g * exp(i %.4g pi)" % (abs(etac_gc), cmath.phase(etac_gc)/cmath.pi))

Part 3b: copper, sigma=5.8e7 S/m, mu=mu0, eps=eps0, f=60 Hz
k^2 = mu eps w^2 + i mu sigma w   (complex)
beta = 117.2   alpha = 117.2   (1/m)
skin depth delta = 1/alpha = 0.008532 m
good-conductor approx: beta=alpha=sqrt(mu sigma w/2) = 117.2,  delta = sqrt(2/(mu sigma w)) = 0.008532 m
check (beta+i alpha)^2 == k^2 : True
check 2 alpha beta == mu sigma w : True
eta_c = w mu/k = 2.858e-06 * exp(i -0.25 pi)  |eta_c| = 2.858e-06 ohm
good-conductor eta_c = sqrt(w mu/sigma) e^{-i pi/4} = 2.858e-06 * exp(i -0.25 pi)


In [13]:
# ---------- Part 3c: energy balance for the lossy plane wave ----------
print("energy balance  -d<S>/dz = <Jf.E>")
b, a, om, mu_s, sig_s, E0s = sp.symbols('beta alpha omega mu sigma E0', positive=True)
Savg_z = b*E0s**2*sp.exp(-2*a*z)/(2*om*mu_s)          # <S_z> = beta |E~|^2 e^{-2 alpha z}/(2 w mu)
JfE_avg = sig_s*E0s**2*sp.exp(-2*a*z)/2                # <Jf.E> = sigma |E~|^2 e^{-2 alpha z}/2
bal = sp.simplify(-sp.diff(Savg_z, z) - JfE_avg)
print("-d<S>/dz - <Jf.E> =", bal)
print("vanishes  <=>  2 alpha beta = mu sigma omega  (the imaginary part of the dispersion relation)")
# stored energy average
uavg = sp.Rational(1, 4)*E0s**2*(eps + (b**2 + a**2)/(om**2*mu_s))*sp.exp(-2*a*z)
print("<u> = 1/4 |E~|^2 (eps + (beta^2+alpha^2)/(w^2 mu)) e^{-2 alpha z}  (lossless limit -> eps|E~|^2/2 e^{-2alpha z})")

energy balance  -d<S>/dz = <Jf.E>
-d<S>/dz - <Jf.E> = E0**2*(alpha*beta - mu*omega*sigma/2)*exp(-2*alpha*z)/(mu*omega)
vanishes  <=>  2 alpha beta = mu sigma omega  (the imaginary part of the dispersion relation)
<u> = 1/4 |E~|^2 (eps + (beta^2+alpha^2)/(w^2 mu)) e^{-2 alpha z}  (lossless limit -> eps|E~|^2/2 e^{-2alpha z})


## 6. What changes when matter enters

| Quantity | Vacuum | Linear medium |
|---|---|---|
| Constitutive relations | $\mathbf{D}=\varepsilon_0\mathbf{E}$, $\mathbf{B}=\mu_0\mathbf{H}$ | $\mathbf{D}=\varepsilon\mathbf{E}$, $\mathbf{B}=\mu\mathbf{H}$, $\mathbf{J}_f=\sigma\mathbf{E}$ |
| Wave equation | $\nabla^2\mathbf{E} = \mu_0\varepsilon_0\,\ddot{\mathbf{E}}$ | $\nabla^2\mathbf{E} = \mu\varepsilon\,\ddot{\mathbf{E}} + \mu\sigma\,\dot{\mathbf{E}}$ (damped) |
| Dispersion | $k^2 = \mu_0\varepsilon_0\omega^2$ | $k^2 = \mu\varepsilon\omega^2 + i\mu\sigma\omega$ (complex $k$) |
| Speed / index | $c$ | $v = 1/\sqrt{\mu\varepsilon}$, $n = c\sqrt{\mu\varepsilon}$ (lossless) |
| Intrinsic impedance | $\eta_0 \approx 377\ \Omega$ | $\eta = \sqrt{\mu/(\varepsilon - i\sigma/\omega)}$ |
| Energy density | $\tfrac12(\varepsilon_0E^2 + B^2/\mu_0)$ | $\tfrac12(\mathbf{E}\cdot\mathbf{D} + \mathbf{B}\cdot\mathbf{H})$ |
| Poynting vector | $\mathbf{E}\times\mathbf{B}/\mu_0$ | $\mathbf{E}\times\mathbf{H}$ |
| Energy balance | $\partial_t u + \nabla\cdot\mathbf{S} = 0$ | $\partial_t u + \nabla\cdot\mathbf{S} = -\sigma\lvert\mathbf{E}\rvert^2$ |
| Attenuation | none | skin depth $\delta = 1/\alpha$; good conductor $\delta = \sqrt{2/(\mu\sigma\omega)}$ |

The **transverse** structure, the relation $\mathbf{B}_0 = \mathbf{k}\times\mathbf{E}_0/\omega$, and the right-handed
$\{\mathbf{E},\mathbf{H},\hat{\mathbf{k}}\}$ triad survive unchanged. Matter only changes the *magnitude* of the field ratio
(via $\eta$), the *speed* (via $n$), and adds *attenuation* (via $\alpha$) and a *phase lag* between $\mathbf{E}$ and $\mathbf{H}$.

**Scope.** This covers linear, isotropic, homogeneous, *nondispersive* media. Real materials are dispersive
($\varepsilon,\sigma$ depend on $\omega$), which makes $u$ and $\mathbf{S}$ frequency-dependent (Brillouin energy formula);
anisotropic media give $\mathbf{D} = \varepsilon_{ij}E_j$ and elliptically polarized waves with $\mathbf{E}$ no longer
strictly transverse to $\mathbf{k}$.

## Appendix: files

| File | Role |
|---|---|
| `maxwell_essay.md` | The full essay (prose version of this notebook) |
| `maxwell_plane_wave.py` | §3 verification script |
| `poynting_derivation.py` | §4 verification script |
| `maxwell_matter.py` | §5 verification script |
| `maxwell_essay.ipynb` | This notebook (executed, outputs embedded) |

To re-execute: `./.venv/Scripts/python.exe -m jupyter nbconvert --to notebook --execute --inplace maxwell_essay.ipynb`
(kernel `maxwell-venv` is registered for the project venv, which has SymPy 1.14).